In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "utils" / "paths.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook inside the project directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import RAW, stage_path
from utils.runs import require_stage, stage_run
from loguru import logger
from rdkit import Chem
from rdkit.Chem import AllChem
from pathlib import Path
from tqdm.auto import tqdm
from itertools import product
import sys
import time
import os
import io
from loguru import logger


from utils import draw

In [ ]:
raw_folder = require_stage("00_scaffold")
BATCH_SIZE = 10000
radicals_pool = {
    "CH3": "C",
    "CH2OH": "CO",
    "OMe": "OC",
    "OH": "O",
    "H": "[H]",
    "NH2": "N",
    "NO2": "[N+](=O)[O-]",
    "NHMe": "NC",
    "SO3H": "S(=O)(=O)O",
    "SO2Me": "S(=O)(=O)C",
    "F": "F",
    "Cl": "Cl",
    "NHMe2": "N(C)C",
    "NHAc": "NC(=O)C",
    "CONH2": "C(=O)N",
    "CONHMe": "C(=O)NC",
    "COOH": "C(=O)O",
    "SO2NH2": "S(=O)(=O)N",
    "NHNH2": "NN",
    "CH2NH2": "CN",
    "CH2OH_2": "C(O)CO",
    "OCH2CH2OH": "OCCO",
    "OCH2CH2NH2": "OCCN",
    "NHCONH2": "NC(=O)N",
}

combinations_count = len(radicals_pool) ** 4
logger.info(f"Rads count: {len(radicals_pool)}")
logger.info(f"Combs count: {combinations_count:,}")

n = len(radicals_pool)

unique_size = (n**2 * (n**2 + 1)) // 2
logger.info(f"Unique combinations: {unique_size:,}")


In [ ]:
with open(raw_folder / "curcumin_scaffold.smi", "r") as f:
    smiles = f.read()

rad_mol = Chem.MolFromSmiles(smiles)
draw(rad_mol)

In [ ]:
def build_analog(base_mol, comb, radicals_pool):
    work_mol = Chem.Mol(base_mol) 
    
    for i, group_key in enumerate(comb):
        group_smiles = radicals_pool.get(group_key)
        
        if not group_smiles:
            logger.error(f"Group {group_key} not found.")
            continue
        
        rxn_smarts = f'[*:1]-[#{100+i}]>>[*:1]-{group_smiles}'
        rxn = AllChem.ReactionFromSmarts(rxn_smarts)
        
        products = rxn.RunReactants((work_mol,))
        
        if products and products[0]:
            work_mol = products[0][0]
        else:
            logger.warning(f"Reaction failed for R-group {group_key}.")
            return None, None 

    try:
        Chem.SanitizeMol(work_mol)
        
        smiles_string = Chem.MolToSmiles(work_mol)
        return smiles_string, comb
        
    except Exception as e:
        logger.error(f"Sanitization or Ro5 error: {e}")
        return None, None

In [ ]:
with stage_run("01_generation", [raw_folder / "curcumin_scaffold.smi"], {"batch_size": BATCH_SIZE, "radicals": radicals_pool}) as output_dir:
    ligands_path = output_dir / "generated_ligands.smi"
    unique = (r for r in product(radicals_pool.keys(), repeat=4) if (r[0], r[1]) <= (r[2], r[3]))
    BATCH_SIZE = 10000

    start_time = time.time()
    counter_total = 0
    counter_written = 0
    buffer = io.StringIO()

    logger.info(f"Starting generation. Batch size: {BATCH_SIZE}")

    try:
        with open(ligands_path, 'w') as outfile:
            for comb in tqdm(unique, total=unique_size):
                smiles, comb = build_analog(rad_mol, comb, radicals_pool)
                counter_total += 1
            
                if smiles:
                    buffer.write(f"""{smiles} {"_".join(comb)}\n""")
                    counter_written += 1
                
                    if counter_written % BATCH_SIZE == 0:
                        outfile.write(buffer.getvalue())
                        buffer.seek(0)
                        buffer.truncate(0)

            outfile.write(buffer.getvalue())

    except Exception as e:
        logger.exception("Generation failed")
        raise

    finally:
        end_time = time.time()
        logger.info("--- FINISHED ---")
        logger.info(f"Total time: {end_time - start_time:.2f} seconds.")
        logger.info(f"Total combinations processed: {counter_total}")
        logger.info(f"Successfully written: {counter_written}")
        logger.info(f"Failed: {counter_total - counter_written}")
